# Legendre-KAN-Conv

Liu et al., *KAN: Kolmogorov-Arnold Networks*, 2024 ([arXiv:2404.19756](https://arxiv.org/abs/2404.19756)); Bodner et al., *Convolutional Kolmogorov-Arnold Networks*, 2024 ([arXiv:2406.13155](https://arxiv.org/abs/2406.13155)).

KAN replaces a linear weight + fixed nonlinearity with a learnable univariate function per edge (originally a B-spline). ConvKAN puts that inside a convolution. Here the per-tap edge function is a degree-K Legendre-polynomial expansion of the (tanh-squashed) input, folded into channels and consumed by one ordinary `Conv2d` -- no single canonical paper covers the Legendre variant specifically; it follows the general ConvKAN pattern used by community implementations (e.g. `torch-conv-kan`'s `ResKANet`). See `model.py`.

Trains on real CIFAR-10.

In [ ]:
import sys
sys.path.insert(0, '../..')
sys.path.insert(0, '.')

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

from cnn_playground.data import load_cifar10
from cnn_playground.device import resolve_device
from cnn_playground.utils.seed import set_seed
from model import LegendreKANModel

set_seed(0)
device = resolve_device('auto')
print('device:', device)

In [ ]:
train_ds = load_cifar10(train=True)
test_ds = load_cifar10(train=False)
train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False)
print(len(train_ds), len(test_ds))

In [ ]:
def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            preds = model(imgs).argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.numel()
    return correct / total

model = LegendreKANModel().to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

history = {'train_loss': [], 'test_acc': []}
epochs = 20
for epoch in range(epochs):
    model.train()
    last_loss = None
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        opt.zero_grad()
        loss = loss_fn(model(imgs), labels)
        loss.backward()
        opt.step()
        last_loss = loss.item()
    history['train_loss'].append(last_loss)
    history['test_acc'].append(evaluate(model, test_loader))

print(f"final test accuracy: {history['test_acc'][-1]:.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(history['train_loss']); axes[0].set_title('train loss'); axes[0].set_xlabel('epoch')
axes[1].plot(history['test_acc']); axes[1].set_title('test accuracy'); axes[1].set_xlabel('epoch')
fig.tight_layout()
plt.show()

In [ ]:
classes = train_ds.classes
imgs, labels = next(iter(test_loader))
imgs, labels = imgs[:6].to(device), labels[:6]
preds = model(imgs).argmax(dim=1).cpu()

mean = torch.tensor([0.4914, 0.4822, 0.4465]).view(3,1,1)
std = torch.tensor([0.2470, 0.2435, 0.2616]).view(3,1,1)

fig, axes = plt.subplots(1, 6, figsize=(12, 2.5))
for i, ax in enumerate(axes):
    img = (imgs[i].cpu() * std + mean).clamp(0,1).permute(1,2,0)
    ax.imshow(img); ax.axis('off')
    ax.set_title(f'pred:{classes[preds[i]]}\ntrue:{classes[labels[i]]}', fontsize=9)
fig.tight_layout()
plt.show()